# 02 · Train the disease classifier (fp16 mixed precision)

Kaggle T4 gives ~2.5 h for 12 epochs on the full 70k-image training set.
Two-stage schedule: 1 epoch head-only, then unfreeze everything at a lower LR.
The number to beat: **val macro-F1**, not accuracy — the classes are imbalanced.


In [ ]:
# Kaggle setup — run this first in every notebook.
# Settings -> Accelerator: GPU T4 x2 | Internet: ON
import os, subprocess, sys
REPO = "/kaggle/working/SKIT-DS-2023-2027-015"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/shankerjoshi/SKIT-DS-2023-2027-015.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxconverter-common", "shap"], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


### Find the CPU sweet spot first
Each DataLoader worker is a separate process decoding JPEGs. Too few starve the GPU; more than the vCPU count oversubscribes and everything slows down. Measure it once, then use the winner.


In [ ]:
import os, time
from torch.utils.data import DataLoader
from src.data.dataset import LeafDataset, build_transforms, index_folder

TRAIN_DIR = "data/fast/plantvillage/train"   # from data/prepare_fast.py; falls back to the raw dir
items, classes = index_folder(TRAIN_DIR)
ds = LeafDataset(items[:4000], build_transforms(True, 224), classes)
print("vCPUs available:", os.cpu_count())

for w in [0, 1, 2, 4]:
    kw = {"prefetch_factor": 4} if w else {}
    dl = DataLoader(ds, batch_size=64, num_workers=w, pin_memory=True, persistent_workers=w > 0, **kw)
    it = iter(dl); next(it)                       # warm-up batch
    t0 = time.perf_counter(); n = 0
    for i, _ in enumerate(it):
        n += 64
        if i == 19:
            break
    print(f"workers={w}: {n / (time.perf_counter() - t0):6.1f} images/s")


Use the best worker count below, and try `--batch-size 128` once — if it does not raise CUDA out-of-memory, it is free throughput.


In [ ]:
!python -m src.train --config configs/train_disease.yaml --batch-size 128


In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
h = pd.DataFrame(json.load(open("runs/history.json")))
display(h)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val")
ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(h.epoch, h.val_macro_f1, label="val macro-F1")
ax[1].plot(h.epoch, h.val_acc, label="val accuracy"); ax[1].legend(); ax[1].set_title("metrics")
plt.tight_layout(); plt.savefig("results_training_curve.png", dpi=110); plt.show()
print("best macro-F1:", h.val_macro_f1.max())


### Before you close this notebook
- Save `runs/best.pt` to the Kaggle output (or push it to a GitHub Release) — it is gitignored on purpose.
- Record the epoch count, batch size, LR and the best macro-F1 in the report table.
